In [1]:
# ============================================================
#  TransAttUNet on BUSI — DGX Multi-GPU Aware Version
#
#  PROBLEM: Other processes are using 77 GB of your 79 GB GPU.
#  SOLUTION:
#    1. Auto-selects the GPU with most free VRAM
#    2. Falls back to CPU if no GPU has enough memory
#    3. Uses IMG_SIZE=256 to reduce memory 4x vs 512
#    4. All previous optimizations kept (AMP + checkpointing)
# ============================================================

# ============================================================
#  STEP 1: Run this FIRST to see GPU status
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi',
                        '--query-gpu=index,name,memory.free,memory.total',
                        '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
print("=" * 60)
print("  GPU STATUS ON THIS DGX")
print("=" * 60)
print(f"  {'GPU':<5} {'Name':<25} {'Free MB':>10} {'Total MB':>10}")
print("-" * 60)

gpu_free = {}
for line in result.stdout.strip().split('\n'):
    if not line.strip():
        continue
    parts   = [p.strip() for p in line.split(',')]
    idx     = int(parts[0])
    name    = parts[1]
    free_mb = int(parts[2])
    total_mb= int(parts[3])
    gpu_free[idx] = free_mb
    bar_len = int(free_mb / total_mb * 20)
    bar     = "█" * bar_len + "░" * (20 - bar_len)
    print(f"  GPU{idx:<4} {name:<25} {free_mb:>8} MB  {total_mb:>8} MB  [{bar}]")

print("=" * 60)

# pick GPU with most free memory
best_gpu  = max(gpu_free, key=gpu_free.get)
best_free = gpu_free[best_gpu]
print(f"\n  ✓ Best GPU: {best_gpu}  ({best_free} MB free)")

if best_free < 4000:
    print("\n  ⚠️  WARNING: All GPUs are nearly full.")
    print("  Options:")
    print("    1. Ask admin to kill idle jobs:  nvidia-smi  then  kill <PID>")
    print("    2. Wait for a GPU to free up")
    print("    3. This script will use IMG_SIZE=128 as last resort\n")

# ============================================================
#  STEP 2: Imports
# ============================================================
import os, glob, random, warnings
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint as grad_checkpoint

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

# ============================================================
#  STEP 3: CONFIG — auto-set based on available VRAM
# ============================================================
DATA_ROOT = r'./Dataset_BUSI_with_GT'
SAVE_DIR  = './checkpoints_transattunet'
EPOCHS    = 50
LR        = 0.03
SEED      = 42

# auto-configure based on free VRAM
if   best_free >= 40000: IMG_SIZE, BATCH_SIZE, TF_LAYERS = 512, 4, 6
elif best_free >= 20000: IMG_SIZE, BATCH_SIZE, TF_LAYERS = 512, 2, 6
elif best_free >= 10000: IMG_SIZE, BATCH_SIZE, TF_LAYERS = 256, 4, 6
elif best_free >=  6000: IMG_SIZE, BATCH_SIZE, TF_LAYERS = 256, 2, 4
elif best_free >=  4000: IMG_SIZE, BATCH_SIZE, TF_LAYERS = 256, 1, 4
else:                    IMG_SIZE, BATCH_SIZE, TF_LAYERS = 128, 1, 2

print(f"\n  Auto config:")
print(f"    IMG_SIZE   = {IMG_SIZE}")
print(f"    BATCH_SIZE = {BATCH_SIZE}")
print(f"    TF_LAYERS  = {TF_LAYERS}")

# ============================================================
#  STEP 4: Set device to best GPU
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.set_device(best_gpu)
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()             # free any leftover allocations
    torch.backends.cudnn.benchmark = True
    device = torch.device(f"cuda:{best_gpu}")
    print(f"\n  Using: cuda:{best_gpu} — "
          f"{torch.cuda.get_device_name(best_gpu)}")
else:
    device = torch.device("cpu")
    print("\n  No GPU available — using CPU")

# ============================================================
#  STEP 5: Verify dataset
# ============================================================
assert os.path.isdir(DATA_ROOT),                              f"❌ Not found: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, 'benign')),     "❌ 'benign' missing"
assert os.path.isdir(os.path.join(DATA_ROOT, 'malignant')), "❌ 'malignant' missing"
print(f"\n  Dataset: {DATA_ROOT}  ✓")

# ============================================================
#  STEP 6: Dataset
# ============================================================
class BUSIDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image    = np.array(Image.open(img_path).convert("RGB"))

        base       = img_path.replace(".png", "")
        mask_files = sorted(glob.glob(base + "_mask*.png"))
        merged     = np.zeros(image.shape[:2], dtype=np.uint8)
        for mf in mask_files:
            m      = np.array(Image.open(mf).convert("L"))
            merged = np.clip(merged + (m > 127).astype(np.uint8), 0, 1)

        if self.transform:
            out   = self.transform(image=image, mask=merged)
            image = out["image"]
            mask  = out["mask"].long()
        else:
            image = torch.from_numpy(
                image.transpose(2, 0, 1)).float() / 255.0
            mask  = torch.from_numpy(merged).long()

        return image, mask


def get_image_paths(root):
    paths = []
    for cls in ["benign", "malignant"]:
        folder = os.path.join(root, cls)
        pngs   = sorted(glob.glob(os.path.join(folder, "*.png")))
        paths += [p for p in pngs if "_mask" not in os.path.basename(p)]
    return paths


def split_80_20(paths, seed=SEED):
    benign    = sorted([p for p in paths
                        if os.sep + "benign"    + os.sep in p])
    malignant = sorted([p for p in paths
                        if os.sep + "malignant" + os.sep in p])
    rng = random.Random(seed)
    rng.shuffle(benign); rng.shuffle(malignant)
    nb = int(len(benign)    * 0.8)
    nm = int(len(malignant) * 0.8)
    return (benign[:nb]  + malignant[:nm],
            benign[nb:]  + malignant[nm:])


all_paths              = get_image_paths(DATA_ROOT)
train_paths, val_paths = split_80_20(all_paths)
print(f"  Total: {len(all_paths)}  "
      f"Train: {len(train_paths)}  Val: {len(val_paths)}")

# ============================================================
#  STEP 7: Augmentations
# ============================================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GaussNoise(p=0.3),
    A.Affine(scale=(0.9, 1.1), translate_percent=0.05,
             rotate=(-10, 10), shear=(-5, 5), p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_loader = DataLoader(
    BUSIDataset(train_paths, train_transform),
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 4,
    pin_memory  = True,
    drop_last   = True,
)
val_loader = DataLoader(
    BUSIDataset(val_paths, val_transform),
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True,
)
print(f"  Train batches: {len(train_loader)}  "
      f"Val batches: {len(val_loader)}")

# ============================================================
#  STEP 8: TransAttUNet Architecture
# ============================================================

class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            ConvBnRelu(in_ch,  out_ch),
            ConvBnRelu(out_ch, out_ch),
        )
    def forward(self, x): return self.net(x)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(
            nn.Conv2d(F_g,  F_int, 1, bias=False),
            nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(
            nn.Conv2d(F_l,  F_int, 1, bias=False),
            nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g_up = F.interpolate(self.W_g(g), size=x.shape[2:],
                             mode="bilinear", align_corners=True)
        return x * self.psi(self.relu(g_up + self.W_x(x)))


class GSAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        mid = max(ch // 8, 1)
        self.avg  = nn.AdaptiveAvgPool2d(1)
        self.max  = nn.AdaptiveMaxPool2d(1)
        self.cfc  = nn.Sequential(
            nn.Conv2d(ch, mid, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1, bias=False))
        self.sconv = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, x):
        ca = torch.sigmoid(self.cfc(self.avg(x)) + self.cfc(self.max(x)))
        x  = x * ca
        sa = self.sconv(torch.cat([x.mean(1, keepdim=True),
                                   x.max(1,  keepdim=True)[0]], dim=1))
        return x * sa


class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4, drop=0.1):
        super().__init__()
        # make sure heads divides dim
        while dim % heads != 0 and heads > 1:
            heads //= 2
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads,
                                           dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hid        = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, hid), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hid, dim), nn.Dropout(drop))
        self.drop  = nn.Dropout(drop)

    def forward(self, x):
        h, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x    = x + self.drop(h)
        x    = x + self.mlp(self.norm2(x))
        return x


class TransformerEncoderMap(nn.Module):
    def __init__(self, channels, num_layers, num_heads=8,
                 mlp_ratio=4, drop=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(channels, num_heads, mlp_ratio, drop)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(channels)

    def _run(self, tokens):
        for blk in self.blocks:
            tokens = blk(tokens)
        return self.norm(tokens)

    def forward(self, x):
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)        # [B, H*W, C]
        tokens = grad_checkpoint(self._run, tokens,
                                 use_reentrant=False)
        return tokens.transpose(1, 2).reshape(B, C, H, W)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2,
                                mode="bilinear", align_corners=True)
        self.attn = AttentionGate(in_ch, skip_ch, max(skip_ch // 2, 1))
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)
        self.gsam = GSAM(out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:],
                              mode="bilinear", align_corners=True)
        skip = self.attn(g=x, x=skip)
        return self.gsam(self.conv(torch.cat([skip, x], dim=1)))


class TransAttUNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=2,
                 base_ch=64, tf_layers=6, tf_heads=8):
        super().__init__()
        c = base_ch
        self.enc1       = DoubleConv(in_ch, c)
        self.enc2       = EncoderBlock(c,    c*2)
        self.enc3       = EncoderBlock(c*2,  c*4)
        self.enc4       = EncoderBlock(c*4,  c*8)
        self.bottleneck = EncoderBlock(c*8,  c*16)
        self.transformer= TransformerEncoderMap(
                              c*16, tf_layers, tf_heads)
        self.dec4 = DecoderBlock(c*16, c*8,  c*8)
        self.dec3 = DecoderBlock(c*8,  c*4,  c*4)
        self.dec2 = DecoderBlock(c*4,  c*2,  c*2)
        self.dec1 = DecoderBlock(c*2,  c,    c)
        self.out  = nn.Conv2d(c, num_classes, 1)
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def _encode(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        return e1, e2, e3, e4, b

    def forward(self, x):
        if self.training:
            e1, e2, e3, e4, b = grad_checkpoint(
                self._encode, x, use_reentrant=False)
        else:
            e1, e2, e3, e4, b = self._encode(x)
        b  = self.transformer(b)
        d4 = self.dec4(b,  e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.out(d1)


# ============================================================
#  STEP 9: Loss
# ============================================================
class SoftDiceLoss(nn.Module):
    def forward(self, logits, targets):
        p   = torch.softmax(logits, dim=1)[:, 1]
        t   = targets.float()
        num = 2.0 * (p * t).sum()
        den = p.pow(2).sum() + t.pow(2).sum()
        return 1.0 - (num + 1e-5) / (den + 1e-5)

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss()
        self.dice = SoftDiceLoss()
    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice(logits, targets)

# ============================================================
#  STEP 10: Metrics
# ============================================================
def compute_metrics(preds_list, masks_list):
    dices, jaccs, precs, senss, specs = [], [], [], [], []
    assds, hds = [], []
    for pred, mask in zip(preds_list, masks_list):
        pred = pred.astype(bool); mask = mask.astype(bool)
        TP = int(( pred &  mask).sum())
        FP = int(( pred & ~mask).sum())
        TN = int((~pred & ~mask).sum())
        FN = int((~pred &  mask).sum())
        dices.append((2*TP) / (2*TP+FP+FN+1e-8))
        jaccs.append(    TP / (  TP+FP+FN+1e-8))
        precs.append(    TP / (  TP+FP    +1e-8))
        senss.append(    TP / (  TP+FN    +1e-8))
        specs.append(    TN / (  TN+FP    +1e-8))
        if pred.any() and mask.any():
            pe = pred & ~binary_erosion(pred)
            me = mask & ~binary_erosion(mask)
            if pe.any() and me.any():
                d1 = distance_transform_edt(~mask)[pe]
                d2 = distance_transform_edt(~pred)[me]
                assds.append((d1.sum()+d2.sum())/(len(d1)+len(d2)))
                hds.append(max(d1.max(), d2.max()))
            else:
                assds.append(np.nan); hds.append(np.nan)
        else:
            assds.append(np.nan); hds.append(np.nan)
    return {
        "Dice (%)":        np.mean(dices)*100,
        "Jaccard (%)":     np.mean(jaccs)*100,
        "Precision (%)":   np.mean(precs)*100,
        "Sensitivity (%)": np.mean(senss)*100,
        "Specificity (%)": np.mean(specs)*100,
        "ASSD":            np.nanmean(assds),
        "HD":              np.nanmean(hds),
    }

# ============================================================
#  STEP 11: LR schedule
# ============================================================
def poly_lr(optimizer, init_lr, cur_iter, total_iter, power=0.9):
    lr = init_lr * ((1.0 - cur_iter / total_iter) ** power)
    for pg in optimizer.param_groups:
        pg["lr"] = max(lr, 1e-6)

# ============================================================
#  STEP 12: Build model
# ============================================================
model     = TransAttUNet(in_ch=3, num_classes=2,
                         tf_layers=TF_LAYERS).to(device)
criterion = CombinedLoss()
optimizer = optim.SGD(model.parameters(),
                      lr=LR, momentum=0.9, weight_decay=5e-4)
scaler    = GradScaler()

n_params = sum(p.numel() for p in model.parameters()
               if p.requires_grad)
print(f"\n  Model      : TransAttUNet")
print(f"  Parameters : {n_params/1e6:.2f} M")
print(f"  IMG_SIZE   : {IMG_SIZE}")
print(f"  BATCH_SIZE : {BATCH_SIZE}")
print(f"  TF_LAYERS  : {TF_LAYERS}")
print(f"  AMP        : ON   |  Grad checkpoint: ON")

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
#  STEP 13: Training loop
# ============================================================
total_iters  = EPOCHS * len(train_loader)
global_iter  = 0
best_dice    = 0.0
best_epoch   = 0
best_metrics = {}

print("\n" + "="*80)
print("Starting TransAttUNet training ...")
print("="*80)

for epoch in range(1, EPOCHS + 1):

    if torch.cuda.is_available() and epoch == 1:
        torch.cuda.reset_peak_memory_stats(device)

    # ── train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for imgs, masks in tqdm(train_loader,
                            desc=f"Ep {epoch:3d}/{EPOCHS} [train]",
                            leave=False):
        imgs  = imgs.to(device,  non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        poly_lr(optimizer, LR, global_iter, total_iters)
        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss  += loss.item()
        global_iter += 1

    train_loss /= len(train_loader)

    if torch.cuda.is_available() and epoch == 1:
        peak = torch.cuda.max_memory_allocated(device) / 1024**3
        print(f"\n  Peak VRAM after epoch 1: {peak:.2f} GB")

    # ── validate ─────────────────────────────────────────────
    model.eval()
    val_loss, all_preds, all_masks_l = 0.0, [], []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader,
                                desc=f"Ep {epoch:3d}/{EPOCHS} [val]  ",
                                leave=False):
            imgs  = imgs.to(device,  non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with autocast():
                logits    = model(imgs)
                val_loss += criterion(logits, masks).item()
            probs = torch.softmax(logits.float(), dim=1)[:, 1]
            preds = (probs > 0.5).cpu().numpy()
            all_preds.extend(preds)
            all_masks_l.extend(masks.cpu().numpy())

    val_loss /= len(val_loader)
    m = compute_metrics(all_preds, all_masks_l)

    print(f"Ep {epoch:3d} | "
          f"TL={train_loss:.4f} VL={val_loss:.4f} | "
          f"Dice={m['Dice (%)']:5.2f}% "
          f"Jacc={m['Jaccard (%)']:5.2f}% "
          f"Prec={m['Precision (%)']:5.2f}% "
          f"Sens={m['Sensitivity (%)']:5.2f}% "
          f"Spec={m['Specificity (%)']:5.2f}% "
          f"ASSD={m['ASSD']:6.2f} "
          f"HD={m['HD']:6.2f}")

    if m["Dice (%)"] > best_dice:
        best_dice    = m["Dice (%)"]
        best_epoch   = epoch
        best_metrics = m.copy()
        ckpt         = os.path.join(SAVE_DIR,
                                    "best_transattunet_busi.pth")
        torch.save(model.state_dict(), ckpt)
        print(f"  ✓ Best model saved → {ckpt}  "
              f"(Dice={best_dice:.2f}%)")

# ============================================================
#  STEP 14: Final report
# ============================================================
PAPER = {
    "Dice (%)":        75.92,
    "Jaccard (%)":     66.88,
    "Precision (%)":   77.35,
    "Sensitivity (%)": 78.73,
    "Specificity (%)": 97.06,
    "ASSD":            21.85,
    "HD":              88.06,
}

print("\n" + "="*65)
print(f"  TABLE 1 — TransAttUNet  |  Best epoch: {best_epoch}")
print("="*65)
print(f"  {'Metric':<20} {'Ours':>10}  {'Paper':>10}  {'Diff':>8}")
print("-"*65)
for k in PAPER:
    ours  = best_metrics.get(k, float('nan'))
    paper = PAPER[k]
    diff  = ours - paper
    flag  = "▲" if diff >= 0 else "▼"
    print(f"  {k:<20} {ours:>10.2f}  {paper:>10.2f}  "
          f"{flag}{abs(diff):>6.2f}")
print("="*65)

  GPU STATUS ON THIS DGX
  GPU   Name                         Free MB   Total MB
------------------------------------------------------------
  GPU0    NVIDIA A100-SXM4-80GB          856 MB     81920 MB  [░░░░░░░░░░░░░░░░░░░░]
  GPU1    NVIDIA A100-SXM4-80GB        70792 MB     81920 MB  [█████████████████░░░]
  GPU2    NVIDIA A100-SXM4-80GB        58358 MB     81920 MB  [██████████████░░░░░░]

  ✓ Best GPU: 1  (70792 MB free)

  Auto config:
    IMG_SIZE   = 512
    BATCH_SIZE = 4
    TF_LAYERS  = 6

  Using: cuda:1 — NVIDIA A100-SXM4-80GB

  Dataset: ./Dataset_BUSI_with_GT  ✓
  Total: 647  Train: 517  Val: 130
  Train batches: 129  Val batches: 33

  Model      : TransAttUNet
  Parameters : 107.58 M
  IMG_SIZE   : 512
  BATCH_SIZE : 4
  TF_LAYERS  : 6
  AMP        : ON   |  Grad checkpoint: ON

Starting TransAttUNet training ...



  Peak VRAM after epoch 1: 12.52 GB


Ep   1 | TL=0.9970 VL=1.1022 | Dice=29.76% Jacc=20.71% Prec=34.94% Sens=49.02% Spec=83.35% ASSD=102.07 HD=268.52
  ✓ Best model saved → ./checkpoints_transattunet/best_transattunet_busi.pth  (Dice=29.76%)


Ep   2 | TL=0.7323 VL=1.1387 | Dice=28.24% Jacc=18.05% Prec=21.50% Sens=70.67% Spec=77.66% ASSD= 87.31 HD=240.89


Ep   3 | TL=0.6541 VL=0.6678 | Dice=49.63% Jacc=38.45% Prec=64.90% Sens=49.85% Spec=98.02% ASSD= 39.96 HD=130.83
  ✓ Best model saved → ./checkpoints_transattunet/best_transattunet_busi.pth  (Dice=49.63%)


...

Ep  46 | TL=0.2045 VL=0.3538 | Dice=74.35% Jacc=64.53% Prec=80.14% Sens=75.42% Spec=97.42% ASSD= 17.91 HD= 80.00
  ✓ Best model saved → ./checkpoints_transattunet/best_transattunet_busi.pth  (Dice=74.35%)


Ep  47 | TL=0.1899 VL=0.3483 | Dice=74.32% Jacc=64.79% Prec=79.14% Sens=75.35% Spec=97.40% ASSD= 19.71 HD= 79.18


Ep  48 | TL=0.1939 VL=0.3593 | Dice=73.64% Jacc=64.12% Prec=80.90% Sens=72.49% Spec=97.78% ASSD= 18.27 HD= 74.19


Ep  49 | TL=0.1956 VL=0.3508 | Dice=74.12% Jacc=64.76% Prec=80.47% Sens=73.85% Spec=97.70% ASSD= 15.86 HD= 69.90


Ep  50 | TL=0.1783 VL=0.3513 | Dice=74.60% Jacc=65.21% Prec=80.25% Sens=74.38% Spec=97.67% ASSD= 19.00 HD= 74.03
  ✓ Best model saved → ./checkpoints_transattunet/best_transattunet_busi.pth  (Dice=74.60%)

  TABLE 1 — TransAttUNet  |  Best epoch: 50
  Metric                     Ours       Paper      Diff
-----------------------------------------------------------------
  Dice (%)                  74.60       75.92  ▼  1.32
  Jaccard (%)               65.21       66.88  ▼  1.67
  Precision (%)             80.25       77.35  ▲  2.90
  Sensitivity (%)           74.38       78.73  ▼  4.35
  Specificity (%)           97.67       97.06  ▲  0.61
  ASSD                      19.00       21.85  ▼  2.85
  HD                        74.03       88.06  ▼ 14.03
